In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/
!unzip /content/drive/MyDrive/images_256.zip

Streaming output truncated to the last 5000 lines.
  inflating: data/images/7B6E.jpg    
  inflating: data/images/7FC2.jpg    
  inflating: data/images/9F12.jpg    
  inflating: data/images/7E16.jpg    
  inflating: data/images/6E86.jpg    
  inflating: data/images/2607D.jpg   
  inflating: data/images/66E3.jpg    
  inflating: data/images/23749.jpg   
  inflating: data/images/5034.jpg    
  inflating: data/images/20307.jpg   
  inflating: data/images/9CCE.jpg    
  inflating: data/images/820E.jpg    
  inflating: data/images/241BA.jpg   
  inflating: data/images/78C9.jpg    
  inflating: data/images/7A9C.jpg    
  inflating: data/images/8271.jpg    
  inflating: data/images/5379.jpg    
  inflating: data/images/5A9C.jpg    
  inflating: data/images/4E2F.jpg    
  inflating: data/images/876D.jpg    
  inflating: data/images/56DE.jpg    
  inflating: data/images/277BF.jpg   
  inflating: data/images/23DF7.jpg   
  inflating: data/images/3A36.jpg    
  inflating: data/images/9312.jpg    

# New Implementation (calculate on GPU to reduce data transfer)

In [4]:
# Use np.matmul to calculate cosine similarity

import os
import random
import re
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torchvision.models as models
import torchvision.transforms as transforms
import cv2

In [7]:
if torch.cuda.is_available():
    print("CUDA is available!")
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
else:
    print("CUDA is not available. Using CPU instead.")
    device = torch.device("cpu")

CUDA is available!


## Choose Resnet model

In [ ]:
def set_deterministic(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Call this function at the beginning of your script
set_deterministic()

# Load pretrained Resnet and modify for grayscale images
model = models.resnet18(pretrained=True)
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)  # Modify to accept 1-channel (grayscale)
model = nn.Sequential(*list(model.children())[:-1])  # Remove the classification head
model.eval()

# Preprocessing transforms with correct resizing and normalization
preprocess = transforms.Compose([
    transforms.Resize((100, 100), interpolation=cv2.INTER_LINEAR),  # Resize with linear interpolation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  # Normalization as requested
])

## Extract features

In [25]:
# Read mapping data
df = pd.read_excel('/content/drive/MyDrive/final_characteristics-v2.xlsx')
df_1 = df[['UNICODE', 'CHAR']].copy()
char_dict = dict(zip(df_1['UNICODE'], df_1['CHAR']))

def extract_features(img_path):
    img = Image.open(img_path).convert('L')  # Open as grayscale
    img_tensor = preprocess(img).unsqueeze(0)  # Preprocess and add batch dimension

    # Move data and model to GPU
    img_tensor = img_tensor.to(device)
    model.to(device)

    with torch.no_grad():
        features = model(img_tensor).squeeze()  # Extract features
    features = features / torch.norm(features)  # Normalize the feature vector
    return features

def process_image(unicode_value, image_folder):
    img_path = os.path.join(image_folder, f"{unicode_value}.jpg")
    if os.path.exists(img_path):
        return unicode_value, extract_features(img_path)
    else:
        print(f"Not found {img_path}!")
        return unicode_value, None

# def process_image(img_path, image_folder):
#     try:
#       filename = os.path.basename(img_path)
#       match = re.match(r'([0-9A-Fa-f]+)\.jpg', filename) # Match unicode hex and .jpg
#       if match:
#           unicode_hex = match.group(1) #get unicode from filename 8B4A.jpg becomes 8B4A

#           unicode_value = f'U+{int(unicode_hex, 16):04X}' # Convert to Unicode format

#           if os.path.exists(img_path):
#             return unicode_value, extract_features(img_path)
#           else:
#               print(f"Not found {img_path}!")
#               return unicode_value, None
#       else:
#         print(f"Could not parse unicode from {img_path}")
#         return None, None
#     except Exception as e:
#         print(f"Error processing {img_path}: {e}")
#         return None, None

def calculate_embeddings_and_similarity(image_folder):
    feature_dict = {}

    # # Get list of all image files in the folder
    # image_paths = [os.path.join(image_folder, f) for f in os.listdir(image_folder) if f.lower().endswith(('.jpg', '.jpeg'))]

    # Use ThreadPoolExecutor for parallel processing
    with ThreadPoolExecutor() as executor:
    #     results = list(executor.map(lambda img_path: process_image(img_path, image_folder), image_paths))
        results = list(executor.map(lambda unicode_value: process_image(unicode_value, image_folder), df_1['UNICODE']))

    # Build feature dictionary
    for unicode_value, feature_vector in results:
        if feature_vector is not None:
            feature_dict[unicode_value] = feature_vector

    if not feature_dict:
        print("No features extracted!")
        return None, None, None

    # Prepare feature matrix
    feature_matrix = torch.stack(list(feature_dict.values())).to(device)  # Move to GPU
    unicode_list = list(feature_dict.keys())

    # Normalize feature vectors (ensure consistency)
    feature_matrix = feature_matrix / torch.norm(feature_matrix, dim=1, keepdim=True)

    # Calculate cosine similarity matrix using np.matmul
    similarity_matrix = torch.matmul(feature_matrix, feature_matrix.T)

    return unicode_list, similarity_matrix, feature_dict

def load_config(config_path):
    config = {}
    with open(config_path, 'r') as file:
        for line in file:
            key, value = line.strip().split('=')
            config[key] = float(value)
    return config

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [26]:
image_folder = '/content/data/images'

unicode_list, similarity_matrix, feature_dict = calculate_embeddings_and_similarity(image_folder)

Streaming output truncated to the last 5000 lines.
Not found /content/data/images/202E9.jpg!
Not found /content/data/images/202EA.jpg!
Not found /content/data/images/202EB.jpg!
Not found /content/data/images/202ED.jpg!
Not found /content/data/images/202EE.jpg!
Not found /content/data/images/202F0.jpg!
Not found /content/data/images/202FE.jpg!
Not found /content/data/images/20300.jpg!
Not found /content/data/images/20302.jpg!
Not found /content/data/images/20306.jpg!
Not found /content/data/images/20318.jpg!
Not found /content/data/images/2031D.jpg!
Not found /content/data/images/20324.jpg!
Not found /content/data/images/20329.jpg!
Not found /content/data/images/20335.jpg!
Not found /content/data/images/20336.jpg!
Not found /content/data/images/20337.jpg!
Not found /content/data/images/20338.jpg!
Not found /content/data/images/20339.jpg!
Not found /content/data/images/2033E.jpg!
Not found /content/data/images/20342.jpg!
Not found /content/data/images/20353.jpg!
Not found /content/data/i

In [27]:
print(unicode_list)

['20016', '20017', '20027', '20028', '2002A', '2002B', '20032', '20033', '20034', '2003A', '2003F', '20040', '20042', '20044', '20046', '20051', '20054', '20059', '2005A', '2005D', '2005F', '2006A', '20075', '20078', '20079', '2007A', '2007B', '2008E', '200AA', '200AB', '200C5', '200CA', '200CC', '200DD', '200E3', '200E4', '200E9', '200EF', '200F1', '200F7', '20100', '2011C', '20126', '20127', '20129', '2012F', '20133', '20136', '2013A', '2013B', '2013C', '2013D', '2014D', '2014E', '20150', '20152', '2015C', '20173', '2017B', '2017C', '2018D', '201B3', '201CD', '201D5', '201D6', '201EE', '201FC', '201FD', '2020B', '2022C', '2022D', '20232', '20233', '2025C', '2025D', '2025E', '2025F', '20260', '20271', '20281', '20299', '2029A', '2029B', '2029C', '2029D', '2029F', '202A1', '202AB', '202C2', '202E5', '202E6', '202FA', '20307', '20326', '20327', '20328', '20363', '20364', '20365', '20366', '2036D', '20375', '2038A', '203A8', '203A9', '203AA', '203AB', '203AC', '203B4', '203DF', '203E0', 

In [28]:
output_file_name = '/content/output/output_top_k_similar_char_resnet18'

## Find top-k similar characters

In [29]:
def find_similar_images(unicode_list, similarity_matrix, feature_dict, top_k=20):
    output_data = []

    if unicode_list is None:
        return None

    for i, input_unicode in enumerate(unicode_list):
        input_char = char_dict.get(input_unicode, "Unknown") # Handle missing chars
        # try:
        #    hex_value = int(input_unicode[2:], 16)
        #    input_char = chr(hex_value)
        # except (ValueError, TypeError):
        #     input_char = "Unknown"  # Handle potential conversion errors

        # Get similarities for the current character
        similarities = similarity_matrix[i]

        # Find indices of top K similar characters (excluding itself)
        indices = torch.argsort(similarities, descending=True)
        similar_indices = indices[indices != i][:top_k]

        # Get character names and scores
        similar_images = []
        for idx in similar_indices:
            similar_char = char_dict.get(unicode_list[idx.item()], "Unknown") # .item() to get Python int
            similar_images.append((similar_char, similarities[idx].item())) # .item() for similarity value

        # for idx in similar_indices:
        #    try:
        #     hex_value = int(unicode_list[idx][2:], 16)
        #     similar_char = chr(hex_value)
        #    except (ValueError,TypeError):
        #      similar_char = "Unknown"
        #    similar_images.append((similar_char, similarities[idx])) # Use the Unicode directly

        # Take top K similar characters, already sorted by indices
        top_k_similar = [char for char, _ in similar_images]


        output_data.append({
            'Input Character': input_char,
            'Top 20 Similar Characters': top_k_similar
        })

    # Export results to a file
    output_df = pd.DataFrame(output_data)
    os.makedirs('/content/output', exist_ok=True)
    output_df.to_excel(f'{output_file_name}.xlsx', index=False)
    output_df.to_csv(f'{output_file_name}.csv', index=False)

    return output_df

In [30]:
if unicode_list:
    result_df = find_similar_images(unicode_list, similarity_matrix, feature_dict)
    print(result_df)

      Input Character                          Top 20 Similar Characters
0                   𠀖  [𠀪, 乓, 𠀗, 𪜀, 詇, 去, 𡊸, 垬, 悏, 连, 共, 兑, 𧦕, 敧, 岦, ...
1                   𠀗  [𠀫, 𠀖, 垆, 井, 屰, 鬼, 妉, 佒, 光, 嗆, 去, 击, 芛, 垁, 贮, ...
2                   𠀧  [𠄧, 恺, 胣, 世, 𡏡, 䏠, 遁, 盪, 齆, 施, 棉, 他, 俓, 𠻁, 𠉱, ...
3                   𠀨  [衱, 忣, 倲, 岋, 𢙇, 𢆕, 𢪳, 散, 𠸒, 敞, 袟, 极, 踩, 𤜯, 𢫕, ...
4                   𠀪  [𠀖, 𡊸, 𠀫, 漖, 甚, 秘, 𨈒, 沌, 𠱌, 𠲧, 唭, 嗆, 昝, 㖪, 楨, ...
...               ...                                                ...
26039               囹  [図, 駋, 匘, 囧, 陯, 槄, 囥, 囶, 糬, 牊, 囨, 顤, 榒, 譍, 鞱, ...
26040               慄  [慄, 愞, 𥻸, 溧, 𢝛, 𤌣, 憟, 楔, 稬, 𢞅, 溗, 𢜞, 懊, 齩, 倮, ...
26041               﨏  [𤥫, 唂, 焀, 𡋥, 培, 棓, 溬, 珔, 邩, 𣑁, 𢬳, 挣, 㙮, 㭲, 浴, ...
26042               﨡  [蛀, 蛙, 蚭, 蚝, 蛻, 𤬸, 𧍍, 䖴, 蜨, 蛭, 𥑥, 蜋, 蚯, 𠻛, 䖨, ...
26043               﨤  [返, 𨒺, 迭, 𨑤, 級, 扱, 迷, 汳, 痍, 𤜯, 遶, 选, 𧲫, 狊, 鼔, ...

[26044 rows x 2 columns]


In [16]:
from google.colab import files

def download_file(filepath):
  """Downloads a file automatically using the google.colab files module"""
  try:
      files.download(filepath)
      print(f"Successfully downloaded: '{filepath}'")
  except Exception as e:
      print(f"Error downloading '{filepath}': {e}")

In [17]:
download_file(f'{output_file_name}.xlsx')
download_file(f'{output_file_name}.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Successfully downloaded: '/content/output/output_top_k_similar_char_resnet18.xlsx'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Successfully downloaded: '/content/output/output_top_k_similar_char_resnet18.csv'
